# 02 · Hybrid RAG, Semantic Provider & pgvector

Domain-filtered hybrid BM25 + TF-IDF retrieval executes offline. Optional OpenAI-compatible pretrained embeddings with pgvector are implemented separately and require credentials/PostgreSQL. Scores are relevance heuristics, not factual confidence.

In [ ]:
%run 01_document_ingestion.ipynb

In [ ]:
import math
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

_TOKEN = re.compile(r"[a-zA-Z0-9][a-zA-Z0-9_+.-]*")
def tokens(s):return _TOKEN.findall(s.lower())

def _bm25(query,docs,k1=1.5,b=.75):
    terms=tokens(query)
    if not docs or not terms:return [0.0]*len(docs)
    tokenized=[tokens(x) for x in docs]; N=len(docs)
    avgl=sum(len(x) for x in tokenized)/N or 1
    df=Counter(t for doc in tokenized for t in set(doc))
    scores=[]
    for doc in tokenized:
        freq=Counter(doc); dl=len(doc); value=0.0
        for t in terms:
            if t not in df:continue
            idf=math.log(1+(N-df[t]+0.5)/(df[t]+0.5))
            value+=idf*freq[t]*(k1+1)/(freq[t]+k1*(1-b+b*dl/avgl))
        scores.append(value)
    return scores

def hybrid_search(project_id,question,domain=None,top_k=5,source_urls=None,source_type=None,exclude_synthetic=False):
    project_id=ensure_project_id(project_id)
    if domain and domain not in DOMAINS:raise ValueError('Unknown domain')
    q=SearchInput(project_id=project_id,question=question,domain=domain,top_k=top_k)
    sql='''SELECT c.*, d.filename, d.source_url, d.source_type FROM chunks c
           JOIN documents d ON c.document_id=d.id
           WHERE c.project_id=? AND d.status='ready' '''
    values=[project_id]
    if domain:sql+=' AND c.domain=?';values.append(domain)
    if source_type:sql+=' AND d.source_type=?';values.append(source_type)
    if exclude_synthetic:sql+=' AND d.source_type != ?';values.append('SYNTHETIC')
    if source_urls is not None:
        if not source_urls:return []
        sql+=' AND d.source_url IN ('+','.join('?' for _ in source_urls)+')'
        values.extend(source_urls)
    with db_connect() as cx:rows=[dict(r) for r in cx.execute(sql,values)]
    if not rows:return []
    texts=[r['content'] for r in rows]
    # Local offline baseline, not a pretrained semantic model.
    word=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,lowercase=True)
    char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=12000)
    try:
        X=hstack([word.fit_transform(texts),char.fit_transform(texts)],format='csr')
        Q=hstack([word.transform([q.question]),char.transform([q.question])],format='csr')
        semantic=cosine_similarity(X,Q).ravel().tolist()
    except ValueError:
        semantic=[0.0]*len(texts)
    lexical=_bm25(q.question,texts)
    sem_order=sorted(range(len(rows)),key=lambda i:-semantic[i])
    lex_order=sorted(range(len(rows)),key=lambda i:-lexical[i])
    sem_rank={v:i+1 for i,v in enumerate(sem_order)}
    lex_rank={v:i+1 for i,v in enumerate(lex_order)}
    # Reciprocal rank fusion; require at least one nonzero matching signal.
    norm=[(1/(60+sem_rank[i]) if semantic[i]>0 else 0)+(1/(60+lex_rank[i]) if lexical[i]>0 else 0)
          for i in range(len(rows))]
    best=sorted((i for i in range(len(rows)) if max(semantic[i],lexical[i])>0),key=lambda i:-norm[i])[:top_k]
    out=[]
    for i in best:
        r=rows[i]
        loc=f"p.{r['page_number']}" if r['page_number'] else 'section unknown'
        out.append(EvidenceHit(chunk_id=r['id'],document_id=r['document_id'],domain=r['domain'],
                filename=r['filename'],page_number=r['page_number'],excerpt=r['content'],
                citation=f"{r['filename']} ({loc}, chunk {r['chunk_index']+1})",source_url=r['source_url'],
                source_type=r['source_type'],
                score=round(norm[i]*30,5)).model_dump())
    return out

print('Offline hybrid retrieval ready')

### Production integration: externally generated embeddings + pgvector
The following adapter runs only if configured. Embeddings are obtained from a real OpenAI-compatible endpoint. Never call the offline TF-IDF baseline a pretrained semantic model. Use a model-specific vector dimension and reindex when changing models.

In [ ]:
class PGVectorAdapter:
    'Real PostgreSQL/pgvector store. Optional psycopg and OPENAI_API_KEY required.'
    def __init__(self,dsn=None,model=None,dimension=None):
        self.dsn=dsn or os.environ.get('PGVECTOR_URL')
        self.model=model or os.environ.get('EMBEDDING_MODEL','text-embedding-3-small')
        self.dimension=int(dimension or os.environ.get('EMBEDDING_DIMENSION','1536'))
        if not self.dsn:raise RuntimeError('PGVECTOR_URL must be configured')
        if self.dimension<8 or self.dimension>2000:raise ValueError('Use 8..2000 dimensions for pgvector HNSW')
        try:import psycopg
        except ImportError as exc:raise RuntimeError('Install psycopg[binary]') from exc
        self.pg=psycopg

    def embed(self,inputs):
        from openai import OpenAI
        if not os.environ.get('OPENAI_API_KEY'):raise RuntimeError('OPENAI_API_KEY is missing')
        client=OpenAI(api_key=os.environ['OPENAI_API_KEY'],
                      base_url=os.environ.get('OPENAI_BASE_URL') or None,
                      timeout=30,max_retries=2)
        r=client.embeddings.create(model=self.model,input=list(inputs))
        vectors=[x.embedding for x in sorted(r.data,key=lambda z:z.index)]
        if any(len(v)!=self.dimension for v in vectors):
            raise ValueError(f'Embedding dimension mismatches schema ({self.dimension})')
        return vectors

    def setup(self):
        with self.pg.connect(self.dsn) as cn:
            with cn.cursor() as c:
                c.execute('CREATE EXTENSION IF NOT EXISTS vector')
                c.execute(f'''CREATE TABLE IF NOT EXISTS group2_vectors(
                    chunk_id TEXT PRIMARY KEY,
                    project_id TEXT NOT NULL, domain TEXT NOT NULL,
                    model TEXT NOT NULL, embedding vector({self.dimension}) NOT NULL)''')
                c.execute('CREATE INDEX IF NOT EXISTS ix_group2_vectors_scope ON group2_vectors(project_id,domain)')
                c.execute('CREATE INDEX IF NOT EXISTS ix_group2_vectors_hnsw ON group2_vectors USING hnsw (embedding vector_cosine_ops)')
            cn.commit()

    def index_project(self,project_id,domain=None):
        pid=ensure_project_id(project_id)
        sql='SELECT id,domain,content FROM chunks WHERE project_id=?'
        args=[pid]
        if domain:sql+=' AND domain=?';args.append(domain)
        with db_connect() as cx:rows=[dict(r) for r in cx.execute(sql,args)]
        if not rows:return 0
        vectors=self.embed([r['content'] for r in rows])
        with self.pg.connect(self.dsn) as cn:
            with cn.cursor() as c:
                for r,v in zip(rows,vectors):
                    val='['+','.join(str(float(x)) for x in v)+']'
                    c.execute('''INSERT INTO group2_vectors(chunk_id,project_id,domain,model,embedding)
                    VALUES(%s,%s,%s,%s,%s::vector)
                    ON CONFLICT(chunk_id) DO UPDATE SET model=EXCLUDED.model,embedding=EXCLUDED.embedding''',
                    (r['id'],pid,r['domain'],self.model,val))
            cn.commit()
        return len(rows)

    def search(self,project_id,question,domain=None,top_k=5):
        pid=ensure_project_id(project_id)
        vec=self.embed([question])[0];v='['+','.join(str(float(x)) for x in vec)+']'
        sql='''SELECT chunk_id, 1-(embedding <=> %s::vector) AS similarity
             FROM group2_vectors WHERE project_id=%s AND model=%s'''
        args=[v,pid,self.model]
        if domain:sql+=' AND domain=%s';args.append(domain)
        sql+=' ORDER BY embedding <=> %s::vector LIMIT %s';args.extend([v,int(top_k)])
        with self.pg.connect(self.dsn) as cn:
            with cn.cursor() as c:c.execute(sql,args);hits=c.fetchall()
        # Return IDs and scores; caller resolves citations from authoritative SQLite metadata.
        return [{'chunk_id':x[0],'similarity':float(x[1])} for x in hits]

print('Optional native pgvector adapter defined; credentials/database not required for local demo')